# Advanced Auto-Encoders

## What are Autoencoders?

Autoencoders are the data encoding techniques based on Unsupervised Artificial Neural Networks. This special type of ANN is trained to encode the data so that in such a way that data is represented in compressed form. The Autoencoders are also trained to decode the data so that, the original data can be reconstructed as far as possible.

## Architecture of Autoencoders

 The architecture for autoencoders are varied. In this section LSTM autoencoders is discussed. LSTM based autoencoders are used to encode and decode the sequence data.

Why sequence data is challenging to process?

- Sequence data are challenging for prediction task because the size of the is not fixed but it varies.
- Also, the temporal series of the data representation make it challenging to extract the features.

So, the building a predictive model to predict the sequence data involve sequence of operation and hence such problems are called as Sequence-to Sequence. Autoencoders comes as the best choice to handle sequence-to-sequence problems.

## Outlier/Anomaly detection using Autoencoders:

Suppose the input data is highly correlated and requires a technique to detect the anomaly or an outlier then, Autoencoders is the best choice. Since, autoencoders can encode the data in the compressed format, they can handle the correlated data.

Let’s train the autoencoders using MNIST data set using simple Feed Forward neural network.

## Simple 6 layered Feed Forward Autoencoders built to train on MNIST data

Once the autoencoders is trained on MNIST data set, an anomaly detection can be done using 2 different images. First one of the images from the MNIST data set is chosen and feed to the trained autoencoders. Since, this image is not an anomaly, the error or loss function is expected to be very low. Next, when some random image is given as test image, the loss rate is expected to be very high as it is an anomaly.

### Installing Packages

In [1]:
%pip install numpy keras matplotlib tensorflow

Note: you may need to restart the kernel to use updated packages.


### Importing Libraries  

In [2]:
import numpy as np
import keras
from keras.datasets import mnist
from keras.models import Sequential, Model
from keras.layers import Dense, Input
from keras import optimizers
from keras.optimizers import Adam
from keras.preprocessing import image

2025-06-19 11:04:31.162969: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-19 11:04:31.166822: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-19 11:04:31.176154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750331071.191474  100562 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750331071.195917  100562 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750331071.208751  100562 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Building Model

In [12]:
import numpy as np
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Dense, Input
from keras.optimizers import Adam

# Load and preprocess MNIST data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
train_x = x_train.reshape(60000, 784) / 255
val_x = x_test.reshape(10000, 784) / 255

# Build the autoencoder using the Functional API
input_layer = Input(shape=(784,))
x = Dense(512, activation='elu')(input_layer)
x = Dense(128, activation='elu')(x)
bottleneck = Dense(10, activation='linear', name="bottleneck")(x)
x = Dense(128, activation='elu')(bottleneck)
x = Dense(512, activation='elu')(x)
output_layer = Dense(784, activation='sigmoid')(x)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(loss='mean_squared_error', optimizer=Adam())

# Train the autoencoder
trained_model = autoencoder.fit(
    train_x, train_x,
    batch_size=1024,
    epochs=10,
    verbose=1,
    validation_data=(val_x, val_x)
)

# Encoder model (up to bottleneck)
encoder = Model(inputs=input_layer, outputs=bottleneck)
encoded_data = encoder.predict(train_x)  # bottleneck representation

# Decoder model (from bottleneck to output)
encoding_dim = 10
encoded_input = Input(shape=(encoding_dim,))
decoder_layer1 = autoencoder.layers[-3](encoded_input)
decoder_layer2 = autoencoder.layers[-2](decoder_layer1)
decoder_output = autoencoder.layers[-1](decoder_layer2)
decoder = Model(encoded_input, decoder_output)

# Reconstruct data
decoded_output = autoencoder.predict(train_x)

2025-06-19 11:13:37.319433: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 188160000 exceeds 10% of free system memory.
2025-06-19 11:13:37.688573: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 188160000 exceeds 10% of free system memory.


Epoch 1/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 82ms/step - loss: 0.1083 - val_loss: 0.0505
Epoch 2/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 0.0466 - val_loss: 0.0365
Epoch 3/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - loss: 0.0350 - val_loss: 0.0298
Epoch 4/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 0.0289 - val_loss: 0.0257
Epoch 5/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 76ms/step - loss: 0.0254 - val_loss: 0.0237
Epoch 6/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 0.0236 - val_loss: 0.0223
Epoch 7/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - loss: 0.0224 - val_loss: 0.0211
Epoch 8/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 0.0212 - val_loss: 0.0202
Epoch 9/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - loss: 0.0204 - val_loss: 0.0194
Epoch 10/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - loss: 0.0196 - val_loss: 0.0189
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


### Anomaly Detection

Reconstruction error is usually between 5 and 30.
Here it 13.544933
Hence not an anomaly.


In [ ]:
import os

img_path = "/workspaces/AIML_BigProjects/3. Training & Advanced Things/Digit1.png"
if os.path.exists(img_path):
	img = image.load_img(img_path, target_size=(28, 28), color_mode="grayscale")
	input_img = image.img_to_array(img) /255.0
	inputs = input_img.reshape(1, 784)
	target_data = autoencoder.predict(inputs)
	dist = np.linalg.norm(inputs - target_data, axis=-1)
	print(dist)
else:
	print(f"File not found: {img_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
[13.544933]


Although the error is 20.058197 but it is anomaly as we can see the picture. Hence for the threshhold of anomaly we have to set it according to our model by training it on multiple images.

In [ ]:
import os

img_path = "/workspaces/AIML_BigProjects/3. Training & Advanced Things/5761601382019629.jpg"
if os.path.exists(img_path):
	img = image.load_img(img_path, target_size=(28, 28), color_mode="grayscale")
	input_img = image.img_to_array(img) /255.0
	inputs = input_img.reshape(1, 784)
	target_data = autoencoder.predict(inputs)
	dist = np.linalg.norm(inputs - target_data, axis=-1)
	print(dist)
else:
	print(f"File not found: {img_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
[20.058197]


: 